In [1]:
# use widgets as matplotlib backend
%matplotlib widget

# imports
from matplotlib import pyplot as plt
import os
import numpy
import os, sys
from time import time
from tqdm.auto import tqdm
from collections import defaultdict
import numpy as np
import matplotlib as mpl

# import the file object for opening kongsberg files
# Note: function and library naming to be discussed
from themachinethatgoesping.echosounders import kongsbergall
from themachinethatgoesping.echosounders import index_functions
from themachinethatgoesping.pingprocessing import filter_pings

#simplify creating figures
mpl.rcParams['figure.dpi'] = 100
close_plots = True

def create_figure(name, return_ax=True):
    if close_plots: plt.close(name)
    fig = plt.figure(name)
    fig.suptitle = name

    if return_ax:
        return fig, fig.subplots()
    return fig

In [5]:
# list of folders with kongsberg .all or .wcd files (subfolders will be scanned as well)

folder = "../../../unittest_data/"

# list raw data files
files = index_functions.find_files(folder, [".all",".wcd"])
cacheFilePaths = index_functions.get_index_paths(file_paths=files)
files.sort()
            
files.sort()
file_name = files[0]
print("files:      ", len(files))

Found 16 files
files:       16


In [6]:
import themachinethatgoesping as ping
print(ping.__file__)

/home/peurban/.local/share/mamba/envs/dev/lib/python3.14/site-packages/themachinethatgoesping/__init__.py


## Open all files
The function 
Notes: 
1. kongsbergall.KongsbergAllFileHandler(files) scanns and indexes all files and provides access to all files like a combined file stream
2. If a .all and a .wcd files with the same name (one .all and one .wcd) are added, they will be matched to a single file
3. It is not possible to add two .all or two .wcd with the same name, even if they are within different folders
4. Note: if the files are not sorted in time, the datagram packages will not be sorted by time either, however it isi simple to sort the pings at a later stage

In [7]:
# Index all files and initialize the file interfaces
# fm will be the accessor
fm = kongsbergall.KongsbergAllFileHandler(files,init=True)

#print some information about the files that where indexed
print(fm)

indexing files ⠐ 100% :00s<00m:00s] [..0871266855321420.all (1/16)]                               
indexing files ⠠ 100% :00s<00m:00s] [..3008643552583898.wcd (16/16)]                                
indexing files ⢀ 100% :00s<00m:00s] [Found: 712 datagrams in 16 files (10MB)]                                         
Initializing ping interface ⢀ 87% :00s<00m:00s] [Done]                                              
KongsbergAllFileHandler
#######################
-
File infos 
-------------               
- Number of loaded .all files: : 8        
- Number of loaded .wcd files: : 8        
- Total file size: :             10.71 MB 

 Detected datagrams 
^^^^^^^^^^^^^^^^^^^^ 
- timestamp_first:  21/08/2012 17:09:42.36 
- timestamp_last:   21/04/2023 17:48:15.35 
- Total:            712                    
- Datagrams [0x30]: 3                      [PUIDOutput]
- Datagrams [0x31]: 9                      [PUStatusOutput]
- Datagrams [0x33]: 3                      [ExtraParameters]
- Datag

## How to access raw datagrams

In [8]:
# print all datagrams
print(fm.datagram_interface)

KongsbergAllDatagramInterface
#############################
-
Detected datagrams 
--------------------- 
- timestamp_first:  21/08/2012 17:09:42.36 
- timestamp_last:   21/04/2023 17:48:15.35 
- Total:            712                    
- Datagrams [0x30]: 3                      [PUIDOutput]
- Datagrams [0x31]: 9                      [PUStatusOutput]
- Datagrams [0x33]: 3                      [ExtraParameters]
- Datagrams [0x41]: 26                     [AttitudeDatagram]
- Datagrams [0x43]: 16                     [ClockDatagram]
- Datagrams [0x47]: 1                      [SurfaceSoundSpeedDatagram]
- Datagrams [0x49]: 16                     [InstallationParametersStart]
- Datagrams [0x4e]: 80                     [RawRangeAndAngle]
- Datagrams [0x4f]: 6                      [QualityFactorDatagram]
- Datagrams [0x50]: 27                     [PositionDatagram]
- Datagrams [0x52]: 36                     [RuntimeParameters]
- Datagrams [0x55]: 16                     [SoundSpeedProfileDatagr

In [9]:
""" 
Access some package and print it (note you should be able to use access 
all variables that are printed can be accessed using get_"varname"() function. 
Try get_ and use tab completition to see which functions are avaliable
"""

package = fm.datagram_interface.datagrams()[10]
print(package)

XYZDatagram
###########
- bytes:               8040     
- stx:                 0x02     
- datagram_identifier: 0x58     [XYZDatagram]
- model_number:        EM2040   [2040]
- date:                20220730 [YYYYMMDD]
- time_since_midnight: 74864023 [ms]

 date/time 
-----------  
- timestamp: 1659.214e⁶   [s]
- date:      30/07/2022   [MM/DD/YYYY]
- time:      20:47:44.023 [HH:MM:SS]

 datagram content 
------------------  
- ping_counter:               63075     
- system_serial_number:       2106      
- heading:                    4334      [0.01° steps]
- sound_speed:                15092     [0.1 m/s steps]
- transmit_transducer_depth:  1.647     [m]
- number_of_beams:            400       
- number_of_valid_detections: 399       
- sampling_frequency:         20.425e³  [Hz]
- scanning_info:              0         
- etx:                        0x03      
- checksum:                   59691     

 processed 
----------- 
- heading:     43.340   [°]
- sound_speed: 1509.200 [m/s]



In [10]:
# You can also access packages by type
# print the first RuntimeParameters datagram
print(fm.datagram_interface.datagrams("RuntimeParameters")[10])

RuntimeParameters
#################
- bytes:               52       
- stx:                 0x02     
- datagram_identifier: 0x52     [RuntimeParameters]
- model_number:        EM2040C  [2045]
- date:                20140213 [YYYYMMDD]
- time_since_midnight: 29561718 [ms]

 date/time 
-----------  
- timestamp: 1392.279e⁶   [s]
- date:      13/02/2014   [MM/DD/YYYY]
- time:      08:12:41.718 [HH:MM:SS]

 datagram content 
------------------             
- ping_counter:                         9664       
- system_serial_number:                 102        
- operator_station_status:              0          
- processing_unit_status:               0          
- bsp_status:                           0          
- sonar_head_or_transceiver_status:     0          
- mode:                                 0b00010110 
- filter_identifier:                    0b01001110 
- minimum_depth:                        0          [m]
- maximum_depth:                        20         [m]
- absorption_coe

In [11]:
# The datagrams function returns a containter like type. 
# Pacakges can be accessed using [index] but it is also possible to e.g. loop through the data using the functions typically used to lists.
# note that also standard slicing logic works

# loop through every 2nd package between index 2 and 5 and compute the average timestamp
timestamps = []
for p in tqdm(fm.datagram_interface.datagrams(skip_data=True)[2:5:2]): #skip_data speeds up looping because it ignores water column data samples
    timestamps.append(p.get_timestamp())

avg = np.nanmean(timestamps)

  0%|          | 0/2 [00:00<?, ?it/s]

In [12]:
# ping has a bunch of more tools avaliable. E.g. convert unixtime to string and the other way around

# print the timestamp as text (use ping tools)
from themachinethatgoesping.tools.timeconv import unixtime_to_datestring

print(unixtime_to_datestring(np.nanmean(timestamps), format="%d-%m-%Y %H:%M:%S")) #try tab completition in the function to access the documentaiton

30-07-2022 20:47:44


In [13]:
# btw. print documentation is also possible like this
help(unixtime_to_datestring)

Help on nb_func in module themachinethatgoesping.tools_nanopy.timeconv:

unixtime_to_datestring = <nanobind.nb_func object>
    unixtime_to_datestring(unixtime: float, fractionalSecondsDigits: int = 0, format: str = '%z__%d-%m-%Y__%H:%M:%S') -> str

    Converts a UNIX timestamp to a formatted date/time string.
    Args:
        unixtime: UNIX timestamp as double (seconds since
                  1970-01-01T00:00:00Z)
        fractionalSecondsDigits: Number of digits for fractional seconds
                                 (default: 0)
        format: Format string (default: "%z__%d-%m-%Y__%H:%M:%S")

    Returns:
        Formatted date/time string

